In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'monospace'

In [ ]:
RESULTS_DIR = "results"
FIGURES_DIR = "../figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

ABLATIONS = [
    ("clf",        r"$\lambda_{\text{clf}}$",         "cellina-clf"),
    ("disc",       r"$\lambda_{\text{disc}}$",        "cellina-disc"),
    ("domain_clf", r"$\lambda_{\text{domain\_clf}}$", "cellina-domain_clf"),
    ("graph",      r"$\lambda_{\text{graph}}$",        "cellina-graph-contrastive"),
]

dfs = {}
for key, _, _ in ABLATIONS:
    path = os.path.join(RESULTS_DIR, f"ablation_{key}.csv")
    if os.path.exists(path):
        dfs[key] = pd.read_csv(path)
        print(f"{key}: {len(dfs[key])} rows")
    else:
        print(f"{key}: NOT FOUND ({path})")

## Figure 1 — F1 scores (4 panels)

In [ ]:
F1_PALETTE = {
    "F1_celltype":      "#4C4CB6",
    "F1_spatial_domain": "#C83636",
}
F1_LABEL_MAP = {
    "F1_celltype":      "celltype",
    "F1_spatial_domain": "spatial domain",
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=False)

for ax_i, (key, xlabel, title) in enumerate(ABLATIONS):
    ax = axes[ax_i]
    if key not in dfs:
        ax.set_title(title + "\n(no data)", fontsize=13)
        continue

    df = dfs[key][dfs[key]["metric"].isin(["F1_celltype", "F1_spatial_domain"])].copy()
    df["lambda_str"] = df["lambda"].map(lambda x: f"{x:.0e}")
    df["label"] = df["metric"].map(F1_LABEL_MAP)

    palette_mapped = {F1_LABEL_MAP[k]: v for k, v in F1_PALETTE.items()}

    sns.stripplot(
        data=df,
        x="lambda_str",
        y="score",
        hue="label",
        palette=palette_mapped,
        dodge=True,
        jitter=True,
        ax=ax,
    )

    ax.set_ylim(0.2, 0.8)
    ax.tick_params(axis='x', labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_title(title, fontsize=13)

    if ax_i == 0:
        ax.set_ylabel("F1 score", fontsize=13)
        ax.legend(title="Label", title_fontsize=11, fontsize=10,
                  loc='lower left').get_title().set_fontweight("bold")
    else:
        ax.set_ylabel("")
        ax.set_yticklabels([])
        ax.get_legend().remove()

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "ablations_f1.svg"), bbox_inches="tight")
plt.show()

## Figure 2 — Marginal log-likelihood (4 panels)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=False)

for ax_i, (key, xlabel, title) in enumerate(ABLATIONS):
    ax = axes[ax_i]
    if key not in dfs:
        ax.set_title(title + "\n(no data)", fontsize=13)
        continue

    df = dfs[key][dfs[key]["metric"] == "marginal_ll"].copy()
    lambda_vals = sorted(df["lambda"].unique())
    data = [df.loc[df["lambda"] == lv, "score"].values for lv in lambda_vals]
    labels = [f"{lv:.0e}" for lv in lambda_vals]

    bp = ax.boxplot(
        data,
        patch_artist=True,
        labels=labels,
        medianprops=dict(color='black', linewidth=2),
    )

    for i, vals in enumerate(data):
        x = np.random.normal(i + 1, 0.04, size=len(vals))
        ax.scatter(x, vals, color="black", s=25, zorder=3)

    ax.tick_params(axis='x', labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_title(title, fontsize=13)

    if ax_i == 0:
        ax.set_ylabel("Marginal LL", fontsize=13)
    else:
        ax.set_ylabel("")
        ax.set_yticklabels([])

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "ablations_mll.svg"), bbox_inches="tight")
plt.show()